# Análise interativa - Solução 2 Template Matching

**Autor:** Manoel Furtado  
**Objetivo:** estudar a contagem de parafusos por template matching implementada em `solucao_2_template_matching_interativo.py`.

Este notebook foi montado para o estudante testar:

1. escolha de uma região de template;
2. resposta do `cv2.matchTemplate`;
3. efeito do limiar de correlação;
4. efeito de escalas diferentes do template;
5. redução de duplicatas com non-max suppression;
6. execução em todas as imagens do Desafio 1.

## 1. Importações e caminhos

A célula localiza a pasta da solução e importa as funções do script original.

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

SCRIPT_NAME = 'solucao_2_template_matching_interativo.py'

def find_solution_dir(script_name):
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / script_name).exists():
            return candidate
    matches = list(start.rglob(script_name))
    if not matches:
        raise FileNotFoundError(f'Não encontrei {script_name} a partir de {start}')
    return matches[0].parent

SOLUTION_DIR = find_solution_dir(SCRIPT_NAME)
DESAFIO_DIR = SOLUTION_DIR.parents[1]
INPUT_DIR = DESAFIO_DIR / 'data' / 'images'
OUTPUT_DIR = SOLUTION_DIR / 'resultados' / 'template_matching'

sys.path.insert(0, str(SOLUTION_DIR))

from solucao_2_template_matching_interativo import annotate, iter_images, match_image, nms, parse_args, run

print('Solução:', SOLUTION_DIR)
print('Imagens:', INPUT_DIR)
print('Saída  :', OUTPUT_DIR)

## 2. Escolha da imagem de referência

Primeiro exibimos a imagem com eixos. Use as coordenadas do gráfico para definir uma caixa que contenha um único parafuso bem isolado.

In [ ]:
image_paths = iter_images(INPUT_DIR)
print('Imagens encontradas:', len(image_paths))
for index, path in enumerate(image_paths):
    print(index, path.name)

reference_index = 0
reference_path = image_paths[reference_index]
reference_bgr = cv2.imread(str(reference_path))
if reference_bgr is None:
    raise ValueError(f'Não consegui ler {reference_path}')

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(reference_bgr, cv2.COLOR_BGR2RGB))
plt.title(f'Referência: {reference_path.name}')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(color='yellow', linewidth=0.4)
plt.show()

print('Dimensão:', reference_bgr.shape)

## 3. Definição do template

Ajuste `template_box = (x, y, w, h)` até o recorte mostrar apenas um parafuso representativo. Esse é o ponto mais importante da solução.

In [ ]:
template_box = (40, 40, 80, 80)

x, y, w, h = template_box
template_bgr = reference_bgr[y:y + h, x:x + w].copy()
if template_bgr.size == 0:
    raise ValueError('Template vazio. Ajuste template_box.')

view = reference_bgr.copy()
cv2.rectangle(view, (x, y), (x + w, y + h), (0, 0, 255), 3)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(view, cv2.COLOR_BGR2RGB))
axes[0].set_title('Template marcado')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(template_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title('Template recortado')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print('template_box:', template_box)
print('template shape:', template_bgr.shape)

## 4. Resposta do matchTemplate em uma escala

O mapa de resposta mostra onde o template se parece mais com a imagem. Regiões claras indicam maior correlação.

In [ ]:
target_index = 0
target_path = image_paths[target_index]
target_bgr = cv2.imread(str(target_path))

scale = 1.0
template_gray = cv2.cvtColor(template_bgr, cv2.COLOR_BGR2GRAY)
target_gray = cv2.cvtColor(target_bgr, cv2.COLOR_BGR2GRAY)
resized_template = cv2.resize(template_gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)
response = cv2.matchTemplate(target_gray, resized_template, cv2.TM_CCOEFF_NORMED)

print('Imagem alvo:', target_path.name)
print('Resposta mínima/máxima:', float(response.min()), float(response.max()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(target_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Imagem alvo')
axes[0].axis('off')
im = axes[1].imshow(response, cmap='magma')
axes[1].set_title('Mapa de correlação')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 5. Parâmetros de detecção

`threshold` filtra respostas fracas. `nms_threshold` controla quanto duas caixas podem se sobrepor antes de uma ser removida.

In [ ]:
threshold = 0.62
nms_threshold = 0.25
scales = '0.75,0.9,1.0,1.1,1.25'
review_score = 0.70

args = parse_args([
    '--input-dir', str(INPUT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--reference-image', str(reference_path),
    '--template-box', ','.join(map(str, template_box)),
    '--threshold', str(threshold),
    '--nms-threshold', str(nms_threshold),
    '--review-score', str(review_score),
    '--scales', scales,
])

matches = match_image(cv2, np, target_bgr, template_bgr, args)
annotated = annotate(cv2, target_bgr, matches)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f'{target_path.name}: {len(matches)} matches')
plt.axis('off')
plt.show()

print('Matches:', len(matches))
print('Scores:', [round(score, 3) for _, score in matches[:20]])

## 6. Entendendo o NMS

Antes do NMS, muitos pontos próximos podem passar do limiar para o mesmo parafuso. O NMS mantém os melhores e remove duplicatas muito sobrepostas.

In [ ]:
single_scale_threshold = threshold
h, w = resized_template.shape[:2]
ys, xs = np.where(response >= single_scale_threshold)
raw_boxes = [(int(x), int(y), int(w), int(h)) for x, y in zip(xs, ys)]
raw_scores = [float(response[y, x]) for x, y in zip(xs, ys)]
keep = nms(np, raw_boxes, raw_scores, nms_threshold)

print('Candidatos antes do NMS:', len(raw_boxes))
print('Candidatos depois do NMS:', len(keep))
print('Melhores scores:', sorted([raw_scores[i] for i in keep], reverse=True)[:10])

## 7. Comparação de thresholds

Aumentar o threshold reduz falsos positivos, mas pode perder parafusos com iluminação ou escala diferente.

In [ ]:
thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]

for value in thresholds:
    test_args = parse_args([
        '--input-dir', str(INPUT_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--threshold', str(value),
        '--nms-threshold', str(nms_threshold),
        '--scales', scales,
    ])
    found = match_image(cv2, np, target_bgr, template_bgr, test_args)
    avg_score = sum(score for _, score in found) / len(found) if found else 0.0
    print(f'threshold={value:.2f} -> count={len(found):2d} avg_score={avg_score:.3f}')

## 8. Comparação de escalas

Quando os parafusos aparecem em tamanhos diferentes, múltiplas escalas aumentam a chance de detecção.

In [ ]:
scale_options = [
    '1.0',
    '0.9,1.0,1.1',
    '0.75,0.9,1.0,1.1,1.25',
    '0.6,0.75,0.9,1.0,1.1,1.25,1.4',
]

for scale_text in scale_options:
    test_args = parse_args([
        '--input-dir', str(INPUT_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--threshold', str(threshold),
        '--nms-threshold', str(nms_threshold),
        '--scales', scale_text,
    ])
    found = match_image(cv2, np, target_bgr, template_bgr, test_args)
    print(f'scales={scale_text:<35} -> count={len(found)}')

## 9. Rodar em todas as imagens sem salvar

Esta célula calcula contagens em memória. É útil para ajustar parâmetros antes de chamar o script completo.

In [ ]:
rows = []
for path in image_paths:
    image = cv2.imread(str(path))
    found = match_image(cv2, np, image, template_bgr, args)
    avg_score = sum(score for _, score in found) / len(found) if found else 0.0
    rows.append({'image': path.name, 'count': len(found), 'avg_score': round(avg_score, 4), 'needs_review': avg_score < review_score or len(found) == 0})

for row in rows:
    print(row)

## 10. Execução do script completo

Esta célula salva as imagens anotadas, o template usado e o CSV de contagens. Por padrão fica desligada.

In [ ]:
execute_run = False

args_run = parse_args([
    '--input-dir', str(INPUT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--reference-image', str(reference_path),
    '--template-box', ','.join(map(str, template_box)),
    '--threshold', str(threshold),
    '--nms-threshold', str(nms_threshold),
    '--review-score', str(review_score),
    '--scales', scales,
])

if execute_run:
    run(args_run)
else:
    print('Execução completa desativada. Mude execute_run = True para salvar resultados.')

## 11. Dicas de interpretação

- Se existem várias caixas no mesmo parafuso, reduza `nms_threshold`.
- Se aparecem muitos falsos positivos, aumente `threshold`.
- Se parafusos reais desaparecem, reduza `threshold` ou amplie `scales`.
- Se o template inclui fundo demais, refaça `template_box` com um recorte mais justo.
- Template matching é explicável, mas sensível a rotação, iluminação e escala.